# Solar filament instance segmentation: end-to-end inference

This notebook turns the competition test images into `submission.csv` with the submitted pipeline (team OAO noir, public PQ 0.37). It does not train and reads no annotation.

**Inputs**
- `FILAMENT_TEST_IMAGES`: directory with the 180 test images (2048 x 2048 H-alpha JPEG).
- `weights/baseline-fold0/`: detector state in two parts and the refiner checkpoint.

**Steps performed by `baseline.predict`**
1. Rebuild the detector state from its two parts and verify the SHA-256 of all tensors and of the refiner file.
2. Resize each image to 1024 x 1024 and run Mask R-CNN (score >= 0.8, mask >= 0.5).
3. Group detections that share pixels, cut a square window with 1.8x context from the original image for each group and resize it to 256 x 256.
4. Refine each crop with the ResNet-18 U-Net; a four-flip average is used only when it preserves the area, component count and overlap of the single pass. Threshold at 0.7.
5. Split merged groups with the detections as seeds when they are clearly distinct, merge remaining overlaps, drop instances under 256 pixels.
6. Retry an image left without any instance once with detector proposal floor 0.001 and score 0.01.
7. Assert that the masks of every image are disjoint and write run-length encoded rows.

`check_submission.py` then verifies that every test image is covered, that no image is empty and that no two masks overlap.

On a Kaggle T4 GPU the run takes about four minutes and reproduces the submitted file byte for byte (1,194 rows, SHA-256 `e1d77f9dd6d1b37df1dc629e186c3f63abcbeba7eab8e1f72d0ca12ffb6ea7d7`).


In [ ]:
from pathlib import Path
import os, subprocess, sys
repo = Path(os.environ.get('FILAMENT_REPO', Path.cwd() if (Path.cwd() / 'baseline/predict.py').is_file() else Path.cwd().parent))
repo = repo.resolve()
test_images = Path(os.environ['FILAMENT_TEST_IMAGES']).resolve()
weights = repo / 'weights' / 'baseline-fold0'
inventory = repo / 'weights' / 'test-inventory.json'
submission = Path(os.environ.get('FILAMENT_OUTPUT', '/kaggle/working/submission.csv' if Path('/kaggle/working').is_dir() else str(Path.cwd() / 'submission.csv')))
cmd = [sys.executable, '-m', 'baseline.predict', '--images', str(test_images),
       '--weights', str(weights), '--inventory', str(inventory), '--output', str(submission),
       '--detector-score', os.environ.get('FILAMENT_DETECTOR_SCORE', '0.8')]
subprocess.run(cmd, cwd=repo, check=True)
subprocess.run([sys.executable, str(repo / 'check_submission.py'), str(submission), '--images', str(test_images)], cwd=repo, check=True)

## Output summary
Number of instances per image and the SHA-256 of the written file.


In [ ]:
import hashlib
import pandas as pd
frame = pd.read_csv(submission)
per_image = frame['filament_id'].str.rsplit('_', n=1).str[0].value_counts()
print('rows', len(frame), '| images', per_image.size, '| instances per image: min', per_image.min(), 'median', per_image.median(), 'max', per_image.max())
print('sha256', hashlib.sha256(submission.read_bytes()).hexdigest())
